In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import scienceplots
plt.style.use(['science', 'no-latex'])

from FastBEMT import Environment, F1A, Plotter, Propeller, Simulation
from FastBEMT.Aeroacoustics import circular_observer_array, uniform_observer_grid

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {DEVICE}")

In [ ]:
RPM = 7000.0
MAP_DOMAIN_SIZE = 5.0
MAP_GRID_SIZE = 10
MAP_METRIC = "ospl"  # Use "oaspl" for the A-weighted map.
VPM_DATA_DIR = Path("../Data/VPM")
LOADINGS_PATH = VPM_DATA_DIR / "loadings_TBS3_N_per_m.pt"
TIMESTAMPS_PATH = VPM_DATA_DIR / "loadings_TBS3_timestamps_s.csv"


def normalize_propeller_geometry(raw_geometry):
    if "airfoil" in raw_geometry:
        return raw_geometry

    r = np.asarray(raw_geometry["r"], dtype=float)
    radial_edges = np.concatenate(
        ([r[0]], 0.5 * (r[:-1] + r[1:]), [r[-1]])
    )
    shift_forward = np.asarray(raw_geometry["shift_forward"], dtype=float)
    shift_up = np.asarray(raw_geometry["shift_up"], dtype=float)

    return {
        "r": r,
        "dr": np.diff(radial_edges),
        "chord": np.asarray(raw_geometry["chord"], dtype=float),
        "twist": np.asarray(raw_geometry["twist"], dtype=float),
        "airfoil": [
            np.asarray(airfoil, dtype=float)
            for airfoil in raw_geometry["airfoils"]
        ],
        "COM_shift": np.column_stack((-shift_forward, shift_up)),
        "n_blades": int(raw_geometry["n_blades"]),
        "tip_radius": float(radial_edges[-1]),
        "hub_radius": float(radial_edges[0]),
    }


with open("../Data/10x7E_50sections.pkl", "rb") as stream:
    geometry = normalize_propeller_geometry(pickle.load(stream))

environment = Environment(
    a_inf=343.0,
    rho=1.225,
    mu=1.81e-5,
    p_ref=20.0e-6,
)
simulation = Simulation(
    revolutions=1,
    timesteps_per_revolution=1,
    device=DEVICE,
)
propeller = Propeller(geometry, environment, simulation)
f1a = F1A(
    propeller=propeller,
    environment=environment,
    loadings=LOADINGS_PATH,
    rpm=RPM,
    source_times=TIMESTAMPS_PATH,
)
print(
    f"Loaded F1A loadings: {tuple(f1a.loads.shape)} N/m "
    f"with {f1a.nt} source timestamps"
)

In [ ]:
map_observers = uniform_observer_grid(
    size=MAP_DOMAIN_SIZE,
    nx=MAP_GRID_SIZE,
    ny=MAP_GRID_SIZE,
)
f1a.run(
    observers=map_observers,
    observer_time_range=simulation.observer_time_range,
    num_observer_times=simulation.num_obs_times,
)

map_ospl = f1a.ospl.cpu().numpy()
map_oaspl = f1a.oaspl.cpu().numpy()
map_level = map_oaspl if MAP_METRIC == "oaspl" else map_ospl
map_units = "dB(A)" if MAP_METRIC == "oaspl" else "dB"

print(f"F1A pressure shape: {tuple(f1a.p_tot.shape)}")
print(
    f"F1A map {MAP_METRIC.upper()} range: "
    f"{np.nanmin(map_level):.2f} to {np.nanmax(map_level):.2f} {map_units}"
)

In [ ]:
plotter = Plotter(propeller)
plotter.plot_acoustic_map(
    grid_size=MAP_GRID_SIZE,
    domain_size=MAP_DOMAIN_SIZE,
    noise_type="f1a",
    metric=MAP_METRIC,
    levels=map_level,
    figsize=(4.5, 3.0),
    cmap="magma",
    contour_levels=[60],
    mirror=True,
)

In [ ]:
POLAR_RADIUS = 1.88
POLAR_OBSERVERS = 37
elevation_deg = np.linspace(-90.0, 90.0, POLAR_OBSERVERS)
polar_observers = circular_observer_array(
    radius=POLAR_RADIUS,
    n_points=POLAR_OBSERVERS,
)

f1a.run(
    observers=polar_observers,
    observer_time_range=simulation.observer_time_range,
    num_observer_times=simulation.num_obs_times,
)

polar_ospl = f1a.ospl.cpu().numpy()
polar_oaspl = f1a.oaspl.cpu().numpy()
polar_frequency = f1a.frequencies.cpu().numpy()
polar_spl = f1a.spl.cpu().numpy()

blade_passing_frequency = RPM * propeller.n_blades / 60.0
bpf_orders = np.array([1.0, 2.0, 3.0])
bpf_indices = np.array(
    [
        int(np.argmin(np.abs(polar_frequency - order * blade_passing_frequency)))
        for order in bpf_orders
    ]
)
bpf_spl = polar_spl[:, bpf_indices]

overall_curves = {
    "OSPL [dB]": polar_ospl,
    "OASPL [dB(A)]": polar_oaspl,
}
bpf_curves = {
    "1st BPF SPL": bpf_spl[:, 0],
    "2nd BPF SPL": bpf_spl[:, 1],
    "3rd BPF SPL": bpf_spl[:, 2],
}
theta = np.deg2rad(elevation_deg)


def plot_level_polar(axis, curves, title):
    peak = max(float(np.max(values)) for values in curves.values())
    floor = peak - 50.0

    for label, values in curves.items():
        radius = np.maximum(values, floor) - floor
        axis.plot(
            theta,
            radius,
            linewidth=2.0,
            marker="o",
            markersize=3,
            label=label,
        )

    axis.set_title(title, fontsize=12)
    axis.set_theta_zero_location("E")
    axis.set_theta_direction(1)
    axis.set_thetamin(-90.0)
    axis.set_thetamax(90.0)
    axis.set_xticks(
        np.deg2rad([-90.0, -60.0, -30.0, 0.0, 30.0, 60.0, 90.0])
    )
    axis.set_xticklabels(
        [
            r"-90$^\circ$" + "\n-X",
            r"-60$^\circ$",
            r"-30$^\circ$",
            r"0$^\circ$",
            r"30$^\circ$",
            r"60$^\circ$",
            r"+90$^\circ$" + "\n+X",
        ]
    )
    level_ticks = np.arange(
        np.ceil(floor / 10.0) * 10.0,
        peak + 1.0,
        10.0,
    )
    axis.set_yticks(level_ticks - floor)
    axis.set_yticklabels([f"{level:.0f} dB" for level in level_ticks])
    axis.tick_params(axis="both", labelsize=12)
    axis.set_ylim(0.0, peak + 2.0 - floor)
    axis.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=1,
        frameon=True,
        edgecolor="none",
        fontsize=12,
    )


figure, polar_axes = plt.subplots(
    1,
    2,
    figsize=(8.5, 4.4),
    subplot_kw={"projection": "polar"},
    constrained_layout=True,
)
plot_level_polar(polar_axes[0], overall_curves, "Overall F1A levels")
plot_level_polar(polar_axes[1], bpf_curves, "BPF harmonic SPL")
plt.show()